# Segment 6 — Adaptive / Foveated 2.5D Grid

Implementation of the adaptive-resolution LiDAR mapping stage.

### Resolution bands
- 0–10 m → 5 cm
- 10–30 m → 10 cm
- 30–60 m → 25 cm
- 60–100 m → 50 cm

The notebook also includes:
- Global 5 cm coordinate alignment
- Adaptive cells
- Semantic aggregation
- Object-aware refinement
- Terrain roughness refinement
- Uniform 5 cm baseline
- Cell/memory/processing benchmarks
- Visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from collections import defaultdict, Counter
import time
import math


In [ ]:
# Configuration

BASE_RESOLUTION = 0.05
MAX_RANGE = 100.0

DISTANCE_BANDS = [
    (10.0, 0.05),
    (30.0, 0.10),
    (60.0, 0.25),
    (100.0, 0.50),
]

DYNAMIC_THRESHOLD = 0.80
OBSTACLE_CONFIDENCE_THRESHOLD = 0.90
IMPORTANT_OBJECT_RESOLUTION = 0.10
TERRAIN_ROUGHNESS_THRESHOLD = 0.15


## 1. Distance-based Resolution Engine

In [ ]:
def get_distance_resolution(distance):
    """Return the base grid resolution for a point distance."""
    if distance < 0:
        raise ValueError("Distance cannot be negative")

    for max_distance, resolution in DISTANCE_BANDS:
        if distance < max_distance:
            return resolution

    return DISTANCE_BANDS[-1][1]


test_distances = [2, 8, 15, 35, 70, 95]

for d in test_distances:
    print(f"{d:5.1f} m -> {get_distance_resolution(d) * 100:.0f} cm")


## 2. Adaptive Cell

In [ ]:
@dataclass
class AdaptiveCell:
    x_index: int
    y_index: int
    resolution: float

    elevation_sum: float = 0.0
    min_height: float = np.inf
    max_height: float = -np.inf

    semantic_scores: dict = field(
        default_factory=lambda: defaultdict(float)
    )

    confidence_sum: float = 0.0
    occupied: bool = False
    dynamic_probability: float = 0.0
    point_count: int = 0
    last_updated: int = 0

    @property
    def elevation(self):
        if self.point_count == 0:
            return 0.0
        return self.elevation_sum / self.point_count

    @property
    def confidence(self):
        if self.point_count == 0:
            return 0.0
        return self.confidence_sum / self.point_count

    @property
    def semantic_class(self):
        if not self.semantic_scores:
            return "UNKNOWN"
        return max(
            self.semantic_scores,
            key=self.semantic_scores.get
        )

    def add_point(
        self,
        z,
        semantic_label="UNKNOWN",
        confidence=1.0,
        dynamic_probability=0.0,
        frame_id=0
    ):
        self.elevation_sum += z
        self.min_height = min(self.min_height, z)
        self.max_height = max(self.max_height, z)

        self.semantic_scores[semantic_label] += confidence
        self.confidence_sum += confidence

        self.dynamic_probability = max(
            self.dynamic_probability,
            dynamic_probability
        )

        self.occupied = True
        self.point_count += 1
        self.last_updated = frame_id


## 3. Global 5 cm Coordinate System

In [ ]:
def world_to_base_coordinate(x, y):
    """Convert world coordinates to global 5 cm base coordinates."""
    bx = int(np.floor(x / BASE_RESOLUTION))
    by = int(np.floor(y / BASE_RESOLUTION))
    return bx, by


def get_alignment_factor(resolution):
    """Number of 5 cm base cells contained in one adaptive cell."""
    return round(resolution / BASE_RESOLUTION)


def get_adaptive_cell_index(x, y, resolution):
    """Return an aligned adaptive-cell index."""
    bx, by = world_to_base_coordinate(x, y)
    factor = get_alignment_factor(resolution)

    cell_x = bx // factor
    cell_y = by // factor

    return cell_x, cell_y


for resolution in [0.05, 0.10, 0.25, 0.50]:
    print(
        f"{resolution:.2f} m -> "
        f"{get_alignment_factor(resolution)} x "
        f"{get_alignment_factor(resolution)} base cells"
    )


## 4. Object-aware and Terrain-aware Refinement

In [ ]:
def calculate_roughness(z_values):
    """Estimate terrain roughness using standard deviation of height."""
    if len(z_values) < 2:
        return 0.0
    return float(np.std(np.asarray(z_values)))


def refine_resolution(
    base_resolution,
    semantic_class="UNKNOWN",
    dynamic_probability=0.0,
    obstacle_confidence=0.0,
    terrain_roughness=0.0
):
    resolution = base_resolution

    if dynamic_probability > DYNAMIC_THRESHOLD:
        resolution = min(
            resolution,
            IMPORTANT_OBJECT_RESOLUTION
        )

    important_classes = {
        "PEDESTRIAN",
        "PERSON",
        "VEHICLE",
        "CAR",
        "BICYCLE",
        "CYCLIST"
    }

    if semantic_class.upper() in important_classes:
        resolution = min(
            resolution,
            IMPORTANT_OBJECT_RESOLUTION
        )

    if obstacle_confidence > OBSTACLE_CONFIDENCE_THRESHOLD:
        resolution = min(
            resolution,
            IMPORTANT_OBJECT_RESOLUTION
        )

    if terrain_roughness > TERRAIN_ROUGHNESS_THRESHOLD:
        resolution = min(resolution, 0.10)

    return resolution


def final_resolution_engine(
    x,
    y,
    semantic_class="UNKNOWN",
    dynamic_probability=0.0,
    obstacle_confidence=0.0,
    terrain_roughness=0.0
):
    distance = np.sqrt(x*x + y*y)

    base_resolution = get_distance_resolution(distance)

    return refine_resolution(
        base_resolution,
        semantic_class=semantic_class,
        dynamic_probability=dynamic_probability,
        obstacle_confidence=obstacle_confidence,
        terrain_roughness=terrain_roughness
    )


## 5. Generate Synthetic LiDAR Data

In [ ]:
def generate_synthetic_lidar(
    num_points=200000,
    max_range=100
):
    rng = np.random.default_rng(42)

    x = rng.uniform(-max_range, max_range, num_points)
    y = rng.uniform(-max_range, max_range, num_points)

    distance = np.sqrt(x*x + y*y)
    mask = distance <= max_range

    x = x[mask]
    y = y[mask]

    # Ground
    z = rng.normal(0, 0.03, len(x))

    # Rough terrain patch
    terrain_mask = (
        (x > 15) &
        (x < 30) &
        (y > -10) &
        (y < 10)
    )

    z[terrain_mask] += (
        0.3 * np.sin(x[terrain_mask])
    )

    # Semantic labels
    labels = np.full(
        len(x),
        "ROAD",
        dtype=object
    )

    # Vehicle
    vehicle_mask = (
        (x > 20) &
        (x < 25) &
        (y > 5) &
        (y < 9)
    )

    labels[vehicle_mask] = "VEHICLE"

    # Pedestrian
    pedestrian_mask = (
        (x > 45) &
        (x < 46) &
        (y > 10) &
        (y < 11)
    )

    labels[pedestrian_mask] = "PEDESTRIAN"

    confidence = rng.uniform(
        0.7,
        1.0,
        len(x)
    )

    dynamic_probability = np.zeros(len(x))

    dynamic_probability[
        labels == "VEHICLE"
    ] = 0.90

    dynamic_probability[
        labels == "PEDESTRIAN"
    ] = 0.95

    return (
        x,
        y,
        z,
        labels,
        confidence,
        dynamic_probability
    )


x, y, z, labels, confidence, dynamic_probability = (
    generate_synthetic_lidar()
)

print("LiDAR points:", len(x))


## 6. Visualize Raw LiDAR

In [ ]:
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    x,
    y,
    c=z,
    s=1
)

plt.colorbar(scatter, label="Elevation (m)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.title("Synthetic LiDAR Point Cloud")
plt.axis("equal")
plt.show()


## 7. Build Adaptive Grid

In [ ]:
def build_adaptive_grid(
    x,
    y,
    z,
    labels,
    confidence,
    dynamic_probability,
    frame_id=0
):
    grid = {}

    for i in range(len(x)):
        px = x[i]
        py = y[i]
        pz = z[i]

        distance = np.sqrt(px*px + py*py)

        if distance > MAX_RANGE:
            continue

        resolution = get_distance_resolution(distance)

        cx, cy = get_adaptive_cell_index(
            px,
            py,
            resolution
        )

        key = (cx, cy, resolution)

        if key not in grid:
            grid[key] = AdaptiveCell(
                x_index=cx,
                y_index=cy,
                resolution=resolution
            )

        grid[key].add_point(
            z=pz,
            semantic_label=labels[i],
            confidence=confidence[i],
            dynamic_probability=dynamic_probability[i],
            frame_id=frame_id
        )

    return grid


start = time.perf_counter()

adaptive_grid = build_adaptive_grid(
    x,
    y,
    z,
    labels,
    confidence,
    dynamic_probability
)

adaptive_time = time.perf_counter() - start

print("Adaptive cells:", len(adaptive_grid))
print("Processing time:", adaptive_time, "seconds")


## 8. Resolution Distribution

In [ ]:
resolution_counts = Counter(
    cell.resolution
    for cell in adaptive_grid.values()
)

print("Resolution distribution")
print("-----------------------")

for resolution in sorted(resolution_counts):
    count = resolution_counts[resolution]
    print(
        f"{resolution * 100:5.0f} cm -> "
        f"{count:,} cells"
    )

print("\nTotal cells:", len(adaptive_grid))


## 9. Convert Adaptive Grid to DataFrame

In [ ]:
def grid_to_dataframe(grid):
    rows = []

    for cell in grid.values():
        rows.append({
            "x_index": cell.x_index,
            "y_index": cell.y_index,
            "resolution": cell.resolution,

            "x": cell.x_index * cell.resolution,
            "y": cell.y_index * cell.resolution,

            "elevation": cell.elevation,
            "min_height": cell.min_height,
            "max_height": cell.max_height,

            "semantic_class": cell.semantic_class,
            "confidence": cell.confidence,

            "occupancy": int(cell.occupied),
            "dynamic_probability": cell.dynamic_probability,

            "point_count": cell.point_count,
            "last_updated": cell.last_updated
        })

    return pd.DataFrame(rows)


adaptive_df = grid_to_dataframe(adaptive_grid)

adaptive_df.head()


## 10. Visualize Adaptive Resolution

In [ ]:
plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    adaptive_df["x"],
    adaptive_df["y"],
    c=adaptive_df["resolution"] * 100,
    s=np.maximum(adaptive_df["resolution"] * 20, 3)
)

plt.colorbar(
    scatter,
    label="Resolution (cm)"
)

plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.title("Adaptive / Foveated Resolution Map")
plt.axis("equal")
plt.show()


## 11. Visualize Semantic Map

In [ ]:
semantic_map = {
    label: i
    for i, label in enumerate(
        adaptive_df["semantic_class"].unique()
    )
}

semantic_values = (
    adaptive_df["semantic_class"]
    .map(semantic_map)
)

plt.figure(figsize=(10, 8))

plt.scatter(
    adaptive_df["x"],
    adaptive_df["y"],
    c=semantic_values,
    s=8
)

plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.title("Adaptive Semantic 2.5D Map")
plt.axis("equal")
plt.show()

print("Semantic classes:")
print(semantic_map)


## 12. Uniform 5 cm Baseline

In [ ]:
def build_uniform_grid(
    x,
    y,
    z,
    resolution=0.05
):
    grid = {}

    for i in range(len(x)):
        distance = np.sqrt(x[i]**2 + y[i]**2)

        if distance > MAX_RANGE:
            continue

        gx = int(np.floor(x[i] / resolution))
        gy = int(np.floor(y[i] / resolution))

        key = (gx, gy)

        if key not in grid:
            grid[key] = {
                "elevation_sum": 0.0,
                "count": 0
            }

        grid[key]["elevation_sum"] += z[i]
        grid[key]["count"] += 1

    return grid


start = time.perf_counter()

uniform_grid = build_uniform_grid(
    x,
    y,
    z,
    resolution=0.05
)

uniform_time = time.perf_counter() - start

print("Uniform 5 cm cells:", len(uniform_grid))
print("Processing time:", uniform_time, "seconds")


## 13. Memory and Cell Reduction Benchmark

In [ ]:
def estimate_memory_bytes(num_cells):
    # Approximate feature-storage estimate.
    # This excludes Python object/dictionary overhead.
    BYTES_PER_CELL = 25
    return num_cells * BYTES_PER_CELL


adaptive_memory = estimate_memory_bytes(len(adaptive_grid))
uniform_memory = estimate_memory_bytes(len(uniform_grid))

cell_reduction = (
    1 -
    len(adaptive_grid) / len(uniform_grid)
) * 100

memory_reduction = (
    1 -
    adaptive_memory / uniform_memory
) * 100

print(
    f"Cell reduction: {cell_reduction:.2f}%"
)

print(
    f"Estimated memory reduction: "
    f"{memory_reduction:.2f}%"
)


## 14. Benchmark Summary

In [ ]:
benchmark = pd.DataFrame({
    "Metric": [
        "Number of cells",
        "Processing time (s)",
        "Estimated memory (MB)"
    ],

    "Uniform 5cm": [
        len(uniform_grid),
        uniform_time,
        uniform_memory / (1024**2)
    ],

    "Adaptive": [
        len(adaptive_grid),
        adaptive_time,
        adaptive_memory / (1024**2)
    ]
})

benchmark


## 15. Far-away Object Refinement Test

In [ ]:
def inspect_far_objects(
    x,
    y,
    labels,
    dynamic_probability
):
    results = []

    for i in range(len(x)):
        distance = np.sqrt(x[i]**2 + y[i]**2)

        if distance < 30:
            continue

        base_resolution = get_distance_resolution(distance)

        final_resolution = refine_resolution(
            base_resolution,
            semantic_class=labels[i],
            dynamic_probability=dynamic_probability[i]
        )

        if final_resolution < base_resolution:
            results.append({
                "distance": distance,
                "object": labels[i],
                "normal_resolution": base_resolution,
                "refined_resolution": final_resolution
            })

    return pd.DataFrame(results)


far_objects = inspect_far_objects(
    x,
    y,
    labels,
    dynamic_probability
)

far_objects.head(20)


## 16. Final Resolution Decision Engine

The final design combines:

**Distance + Object Importance + Terrain Complexity + Confidence → Resolution → Adaptive Map**

In [ ]:
# Example decisions

examples = [
    {
        "name": "Near road",
        "x": 5,
        "y": 2,
        "semantic": "ROAD",
        "dynamic": 0.0,
        "confidence": 0.95,
        "roughness": 0.02
    },
    {
        "name": "Far road",
        "x": 70,
        "y": 0,
        "semantic": "ROAD",
        "dynamic": 0.0,
        "confidence": 0.95,
        "roughness": 0.02
    },
    {
        "name": "Far pedestrian",
        "x": 50,
        "y": 10,
        "semantic": "PEDESTRIAN",
        "dynamic": 0.95,
        "confidence": 0.95,
        "roughness": 0.02
    },
    {
        "name": "Rough terrain",
        "x": 50,
        "y": 10,
        "semantic": "ROAD",
        "dynamic": 0.0,
        "confidence": 0.95,
        "roughness": 0.30
    }
]

for item in examples:
    resolution = final_resolution_engine(
        item["x"],
        item["y"],
        semantic_class=item["semantic"],
        dynamic_probability=item["dynamic"],
        obstacle_confidence=item["confidence"],
        terrain_roughness=item["roughness"]
    )

    print(
        f'{item["name"]:20s} -> '
        f'{resolution * 100:.0f} cm'
    )


## 17. Save the Adaptive 2.5D Map

In [ ]:
output_file = "adaptive_2_5D_map.csv"

adaptive_df.to_csv(
    output_file,
    index=False
)

print(f"Saved: {output_file}")


## Segment 6 Completion Check

The notebook should now produce:

1. A distance-adaptive 2.5D grid
2. Aligned 5 cm / 10 cm / 25 cm / 50 cm spatial levels
3. Semantic and elevation information per cell
4. Object-aware refinement
5. Terrain-aware refinement
6. Uniform-vs-adaptive benchmark
7. Resolution distribution
8. CSV output for the next segment

### Next integration point

The adaptive grid can now be connected to **Segment 7**, where the vehicle moves and the map is incrementally updated instead of rebuilding the entire map every frame.